In [2]:
import os

os.environ["REQUESTS_CA_BUNDLE"] = "./certs/pwc-bundle.pem"
os.environ["SSL_CERT_FILE"] = "./certs/pwc-bundle.pem"

In [3]:
import pandas as pd

pd.set_option("display.max_colwidth", None)   # show full cell content
pd.set_option("display.max_rows", None)       # optional, show all rows

In [3]:
import polars as pl

df = pl.read_parquet('hf://datasets/lang-uk/recruitment-dataset-candidate-profiles-english/data/train-00000-of-00001.parquet')


In [ ]:
import polars as pl

df = pl.read_parquet('hf://datasets/lang-uk/recruitment-dataset-candidate-profiles-english/data/train-00000-of-00001.parquet')
example_bio_data = df["Moreinfo"]

CV
str
"""Landed a role of Director of B…"
""" Worked on a mobile applicatio…"
""" 1 am an 1C developer. I deplo…"
""" Perfect knowledge of 1C:Ente…"
"""thousands of resolved problems…"
…
""" -Providing legal support to m…"
""" Supported IT startups in esta…"
"""I’m a Lawyer, who is intereste…"


In [4]:
import pandas as pd
import numpy as np

ID_COL = "TalentLink ID"

df = pd.read_excel("Copy of Staff_TalentLink_Profile_Detail2026-2-25-11-56-30-548232807.xlsx", header=6)

# Clean blanks only (no row deletion except fully empty)
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.dropna(how="all")

# Trim strings
obj_cols = df.select_dtypes(include=["object", "string"]).columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.astype("string").str.strip())

# ID as numeric (important for grouping)
df[ID_COL] = pd.to_numeric(df[ID_COL], errors="coerce").astype("Int64")

# Parse job dates safely
for c in ["Job History Start Date", "Job History End Date"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

df.head(2)

C:\Users\dadenuga001\AppData\Local\Temp\ipykernel_26788\3772032856.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")
C:\Users\dadenuga001\AppData\Local\Temp\ipykernel_26788\3772032856.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[c] = pd.to_datetime(df[c], errors="coerce")


,Local employee ID,TalentLink ID,Resource Name,Resource Status,Resource Type,Management level,Job level,Grade Code,Global LoS,Global Network,...,Area of Study,Academic Institution,In Process,Project Type Interest,Travel Interest,Additional Information,Profile Last Edited (YYYY/MM/DD),Unnamed: 54,Unnamed: 55,Unnamed: 56
0,100638903,147651,"Adams, Stephen K",Active,Employee,Senior Manager,Senior Manager,203711,Assurance,Assurance,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2025-04-23,r100638903,46076.499774,|Table2Resource status in 'Active'| |Table2Local sub LoS 2 in 'GBR-Risk_FS'| |Table2Local sub LoS 3 in 'GBR-Risk_FS_TDR'| |Table2PwC territory in 'PwC United Kingdom'|
1,100638903,147651,"Adams, Stephen K",Active,Employee,Senior Manager,Senior Manager,203711,Assurance,Assurance,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2025-04-23,r100638903,46076.499774,|Table2Resource status in 'Active'| |Table2Local sub LoS 2 in 'GBR-Risk_FS'| |Table2Local sub LoS 3 in 'GBR-Risk_FS_TDR'| |Table2PwC territory in 'PwC United Kingdom'|


In [5]:
skills_per_id = df[df["Skill Level 1"].isin(["Skills"])]

skills_per_id_grouped = (
    skills_per_id.groupby("TalentLink ID", sort=False)["Capability Name"]
        .agg(list)
        .reset_index(name="skills")
)

skills_per_id_grouped.head()

,TalentLink ID,skills
0,147651,"[Application-Specific Integrated Circuit (ASIC), Asset Management, Bank Regulatory Reporting, Banking Capital Markets, Behavioral Economics, Business Requirements Documentation (BRD), Capital Markets, Central Banks, Clearing Settlements, Client Work, Computer Science, Customer Analysis, Data Analysis, Data Mining, Data Profiling, Data Quality, Data Visualization, Debt, Debt Advisory, Derivative Contracts, Economics, Equity Research, Equity Securities, Establishing Regulation, Exchange Traded Derivatives, Financial Budgeting, Financial Modeling, Financial Transactions, Foreign Exchange, Foreign Exchange Hedging, Foreign Exchange Risk, Hedging Strategy, Insurance, IPO Preparation, Lloyds (Insurance Market), Management Information (MI) Reporting, Mergers and Acquisitions (M&A), Microsoft Excel, Microsoft SQL Server, MiFID, MiFID II, Predictive Analytics, Predictive Modelling, Python (Programming Language), QlikView (Software), R (Programming Language), Regulatory Compliance, Regulatory Guidelines, Regulatory Processes, Regulatory Reporting, Regulatory Response, Regulatory Risk, Remediation Design, Reporting and Dashboards, Technical Specifications, Transaction Reporting, Treasury Management, Visual Basic, Waterfall Model, Wealth Management]"
1,17759722,"[Accepting Feedback, Active Listening, Alteryx (Automation Platform), Communication, Dashboard Creation, Microsoft Power Business Intelligence (BI), Microsoft Power Query, Optimism, Power BI, Structured Query Language (SQL)]"
2,18096852,"[Analytical Reasoning, Communication, Teamwork, Time Management]"
3,279631,"[Accepting Feedback, Active Listening, Amazon Web Services (AWS), Analytical Thinking, Artificial Intelligence, Banking Capital Markets, C-Level Presentations, Cloud Architectures, Cloud Computing, Cloud Migration, Cloud Security, Cloud Strategy, Coaching and Feedback, Common Business-Oriented Language (COBOL) Programming Language, Communication, Compliance Frameworks, Compliance Policies, Compliance Program Implementation, Compliance Review, Controls Testing, Creativity, Critical Infrastructure Protection (CIP), Customer Information Control System (CICS), Cyber Incident Response, Cyber Threat Analysis, Cybersecurity, Cybersecurity Strategy, Data Security, Disaster Recovery, Embracing Change, Emotional Regulation, Empathy, External Audit, Fraud Detection, Fraud Prevention, Group Facilitation, IBM Db2, IBM Mainframes, Identity and Access Management (IAM), Inclusion, Influence, Innovation Processes, Intellectual Curiosity, Internal Audit, Internal Controls, IT Audit, IT Controls, IT General Controls (ITGC), IT Security Administration, Job Control Language, Learning Agility, Machine Learning, Mainframe Computing, Mainframe Development, Mainframe Operating System, Operating Effectiveness Review, Optimism, Organizational Structure, Policy Reviews, Process Mapping, Professional Courage, Rapid Experimentation, Regulatory Compliance Consulting, Relationship Building, Resource Access Control Facility (RACF), Risk Analysis, Risk Compliance, Risk Governance, Risk Management, Secure Configuration, Security Architecture, Security Governance, Security Policies, Security Standards, Self-Awareness, Service Excellence, SOX Testing, Storytelling, Strategic Questioning, System Auditing, Teamwork, Technology Administration, Threat and Vulnerability Management, Threat Management, Well Being]"
4,21145412,"[Accepting Feedback, Active Listening, Adobe XD, Agile Methodology, Analytical Thinking, Business Development, Change Management, Compliance Review, Compliance Training, Controls Testing, Creativity, Cross-Functional Collaboration, Customer Success, Cyber Risks, Data Governance, Digital Transformation, Embracing Change, Empathy, Employee Onboarding, Figma (UI Design Software), Financial Services, Fraud Detection, Fraud Prevention, Google Analytics, Google Workspace, Inclusion, Information Architecture, Information Security, Information Security Assessments, Info

In [6]:
#This works
language = df[df["Skill Level 1"].isin(["Language"])]

langauage_grouped = (
    language.groupby("TalentLink ID", sort=False)["Capability Name"]
    .agg(list)
    .reset_index(name="languages")
)

langauage_grouped.head()

,TalentLink ID,languages
0,147651,"[English, Greek]"
1,17759722,[English]
2,18096852,[Yoruba]
3,279631,"[English, Spanish]"
4,2841732,"[English, Arabic]"


In [7]:
#This works
summary = df.drop_duplicates(subset=['TalentLink ID', 'Summary/bio'])

biography = (
    summary
        .groupby("TalentLink ID", sort=False)["Summary/bio"]
        .agg(list)
        .reset_index(name="Biography")
)

biography[biography["TalentLink ID"] == 97817]

,TalentLink ID,Biography
98,97817,"[Pravieena specialises in Agile Project Management for Financial Sector clients. Pravieena has experience of working on large technology transformation programmes. Pravieena has experience in designing, implementing and delivering a PMO function, working both with internal and client stakeholders.]"


In [8]:
import pandas as pd
from itertools import zip_longest

# --- 0. columns we care about
cols = [
    "Job History Start Date",
    "Job History End Date",
    "Position",
    "Company",
    "Description"
]

# --- 1. select, dedupe, and copy
jobs_by_id = (
    df[[
        "TalentLink ID",
        *cols
    ]]
    .dropna(subset=["Description"])
    .drop_duplicates()
    .copy()
)

# --- 2. clean text columns safely
text_cols = ["Position", "Company", "Description"]
for c in text_cols:
    jobs_by_id.loc[:, c] = (
        jobs_by_id[c].astype(str)
        .str.replace(r"\n+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"": None})
    )

# --- 3. parse & format dates keeping gaps as None
for c in ["Job History Start Date", "Job History End Date"]:
    jobs_by_id.loc[:, c] = pd.to_datetime(jobs_by_id[c], errors="coerce")
    formatted = jobs_by_id[c].dt.strftime("%Y-%m-%d")
    jobs_by_id.loc[:, c] = formatted.where(jobs_by_id[c].notna(), None)

# --- 4. optional sort so arrays are chronological
jobs_by_id = jobs_by_id.sort_values(
    ["TalentLink ID", "Job History Start Date", "Job History End Date"],
    na_position="last"
)

# --- 5. make parallel lists per ID
grouped = (
    jobs_by_id
      .groupby("TalentLink ID", sort=False)[cols]
      .agg(list)
      .reset_index()
)

# --- 6. convert parallel lists -> list-of-dicts (aligned; zip_longest protects against mismatched lengths)
def lists_to_jobs(row):
    lists = [row[c] for c in cols]
    jobs = []
    for start, end, pos, comp, desc in zip_longest(*lists, fillvalue=None):
        jobs.append({
            "Job History Start Date": start,
            "Job History End Date": end,
            "Position": pos,
            "Company": comp,
            "Description": desc
        })
    return jobs

grouped["Jobs"] = grouped.apply(lists_to_jobs, axis=1)

# --- 7. final frame: one row per ID with Jobs = list of aligned job dicts
projects = grouped[["TalentLink ID", "Jobs"]]

# explode for everyone while keeping TalentLink ID
person_jobs = projects.explode("Jobs").reset_index(drop=True)

# normalize the dict column (skip rows where Jobs is missing)
jobs_df = pd.concat(
    [
        person_jobs[["TalentLink ID"]],
        pd.json_normalize(person_jobs["Jobs"])
    ],
    axis=1
)


I think i should join up all the datasets now to a new big dataset.

In [9]:
master_dataset = (
    skills_per_id_grouped.set_index("TalentLink ID")
    .join(biography.set_index("TalentLink ID"))
    .join(jobs_df.set_index("TalentLink ID"))
)

In [10]:
master_dataset.to_csv("master_dataset.csv", index=True)

In [11]:
#This works
riskTeams = df[['TalentLink ID', 'Strategic Region']]
team_groups = riskTeams.drop_duplicates(subset=['TalentLink ID', 'Strategic Region'])


team = (
    team_groups
        .groupby("TalentLink ID", sort=False)["Strategic Region"]
        .agg(list)
        .reset_index(name="Team")
)

team.head(3)


,TalentLink ID,Team
0,147651,[TDA Banking]
1,17759722,[TDA Tech Risk]
2,18096852,[TDA Insurance]


Now i think i have all the data i really need to build the machine learning model. Its time to start small. I need to find a way to transform text into numbers. This means i need to look at the projects and biography vairables and see if i can extract more info

I think i will use the 4 steps i found in my preliminary research

In [12]:
master_dataframe = (
    master_dataset
    .groupby("TalentLink ID")
    .agg({
        "skills": "first",
        "Biography": "first",
        "Job History Start Date": list,
        "Job History End Date": list,
        "Position": list,
        "Company": list,
        "Description": list
    })
    .reset_index()
)

In [13]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

nlp = spacy.load("en_core_web_sm")


Stop word reomval

In [14]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# load spaCy model
nlp = spacy.load("en_core_web_sm")

# =========================================================
# STEP 1: spaCy stop word removal
# =========================================================
def remove_stopwords(text):
    text = str(text)
    doc = nlp(text)
    tokens = [token.text for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(tokens)

master_dataframe["step1_stopwords_removed"] = master_dataframe["Description"].apply(remove_stopwords)

# =========================================================
# STEP 2: spaCy POS tagging + lemmatization
# =========================================================
def tag_and_lemmatize(text):
    text = str(text)
    doc = nlp(text)

    lemmas = []
    pos_tags = []

    for token in doc:
        if not token.is_stop and not token.is_punct:
            lemmas.append(token.lemma_)
            pos_tags.append((token.text, token.pos_))

    return pd.Series([
        " ".join(lemmas),   # cleaned lemmatized text
        pos_tags            # token + POS tags
    ])

master_dataframe[["step2_lemmatized", "pos_tags"]] = master_dataframe["step1_stopwords_removed"].apply(tag_and_lemmatize)

# =========================================================
# STEP 3: TF-IDF vector representation
# =========================================================
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(master_dataframe["step2_lemmatized"])

# Optional: group by TalentLink ID
grouped_df = (
    master_dataframe
    .groupby("TalentLink ID")["step2_lemmatized"]
    .apply(" ".join)
    .reset_index()
)

X_grouped_tfidf = tfidf_vectorizer.fit_transform(grouped_df["step2_lemmatized"])

In [15]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# group employee text
employee_df = (
    master_dataframe
    .groupby("TalentLink ID", as_index=False)
    .agg({
        "step2_lemmatized": " ".join,
        "skills": "first"
    })
)

# create TF-IDF matrix
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X = vectorizer.fit_transform(employee_df["step2_lemmatized"])
feature_names = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(X.toarray(), columns=feature_names)
tfidf_df["TalentLink ID"] = employee_df["TalentLink ID"]

# score only the employee's listed skills
skill_scores = []

for i, row in employee_df.iterrows():
    employee_id = row["TalentLink ID"]
    employee_skills = row["skills"]

    if not isinstance(employee_skills, list):
        employee_skills = []

    for skill in employee_skills:
        skill_lower = str(skill).lower()
        score = tfidf_df.loc[i, skill_lower] if skill_lower in tfidf_df.columns else 0

        skill_scores.append({
            "TalentLink ID": employee_id,
            "skill": skill,
            "score": score
        })

skill_scores_df = pd.DataFrame(skill_scores)

In [16]:
# keep only skills that appear in descriptions
filtered = skill_scores_df[skill_scores_df["score"] > 0]

skill_analysis_table = (
    filtered
    .groupby("skill")
    .agg(
        employees_demonstrating=("TalentLink ID", "nunique"),
        average_score=("score", "mean"),
        max_score=("score", "max")
    )
    .sort_values("employees_demonstrating", ascending=False)
    .reset_index()
)

skill_analysis_table.head()

,skill,employees_demonstrating,average_score,max_score
0,Risk Management,16,0.048042,0.127513
1,Business Continuity,11,0.064255,0.160426
2,Communication,11,0.020496,0.047358
3,Technology,10,0.031819,0.115997
4,Insurance,8,0.029901,0.086396


What i need to do here is add biography/summary as part of the calculation 

In [18]:
filtered.head()

,TalentLink ID,skill,score
44,103419,Investment Bank,0.084569
85,128641,Business Continuity,0.068386
92,128641,Communication,0.019065
97,128641,Design,0.117735
105,128641,Governance Risk,0.027000
